# TradeLog · Predicting Next-Day Direction of Bitcoin & Tech Stocks

**Final project — מסלול 2: למידת מכונה (מודל לחיזוי מחירי מניות או ביטקוין)**

This notebook is the Data-Science deliverable behind the **Predict** page of [TradeLog](https://github.com/Benru1503/tradelog), our trading-diary web app. It walks the full classic pipeline — data collection, feature engineering, model training, evaluation — and ends with the question the course asks explicitly: *does the strategy derived from the model generate positive alpha net of fees?*

| Course requirement | Where it happens |
|---|---|
| ≥ 3 years of historical data | §1 — daily closes since 2020-01-01 (~6.5 years), 14 assets |
| Merge with alternative data | §2 — Bitcoin on-chain metrics (blockchain.info), macro indices (S&P 500, VIX, DXY), optional Google Trends |
| Build & compare ≥ 2 models | §5–§7 — XGBoost (2 variants) vs an LSTM neural network |
| Detailed Colab notebook (EDA, training, inference) | §3 EDA · §5–§6 training · §9 inference demo |
| Financial success metrics + backtest with fees | §8 — long/flat strategy, 10 bps per position change, vs buy & hold |

**Runtime:** ~8–12 min on a free Colab CPU runtime, fully keyless (no API secrets anywhere, per the submission rules). Run with *Runtime → Run all*.

> ⚠️ Educational project. Nothing here is financial advice — §8 makes it very clear why.

## 0 · Setup

Colab ships pandas/numpy/scikit-learn/TensorFlow/matplotlib. We only add `xgboost` (and optionally `pytrends`).

In [ ]:
%pip -q install xgboost>=2.0
%pip -q install pytrends  # optional — Google Trends section degrades gracefully without it

In [ ]:
import json, math, urllib.request
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import roc_auc_score, accuracy_score

plt.rcParams["figure.figsize"] = (11, 4)
np.random.seed(42)

DATA_START  = "2020-01-01"   # > 3 years of history, as required
VALID_START = "2025-01-01"   # chronological split — no shuffling, ever
TEST_START  = "2025-10-01"

ASSETS = {
    # name: (yahoo symbol, is_crypto)
    "BTC": ("BTC-USD", True),  "ETH": ("ETH-USD", True),
    "AAPL": ("AAPL", False), "MSFT": ("MSFT", False), "NVDA": ("NVDA", False),
    "GOOGL": ("GOOGL", False), "AMZN": ("AMZN", False), "META": ("META", False),
    "TSLA": ("TSLA", False), "AMD": ("AMD", False), "JPM": ("JPM", False),
    "XOM": ("XOM", False), "SPY": ("SPY", False), "QQQ": ("QQQ", False),
}

HORIZONS = {"d1": 1, "w1": 5}   # bars ahead: next day / next week (5 sessions)

## 1 · Data collection — 6.5 years of daily closes, keyless

We pull daily closes + volume from Yahoo Finance's public v8 chart endpoint with plain `urllib` — **the exact same endpoint the production app hits at inference time** (`src/lib/ml/history.ts`), which kills a whole class of train/serve skew.

Two conventions are pinned here and mirrored in the app:
1. A bar whose UTC date is *today* is a session in progress, not a close → dropped.
2. Yahoo's `close` is split-adjusted (not dividend-adjusted) — fine for direction labels, and consistent between training and serving.

In [ ]:
def fetch_yahoo(symbol: str, start: str = DATA_START) -> pd.DataFrame:
    period1 = int(datetime.strptime(start, "%Y-%m-%d").replace(tzinfo=timezone.utc).timestamp())
    period2 = int(datetime.now(timezone.utc).timestamp())
    url = (f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
           f"?period1={period1}&period2={period2}&interval=1d")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (tradelog-notebook)"})
    with urllib.request.urlopen(req, timeout=30) as res:
        data = json.loads(res.read().decode())
    r = data["chart"]["result"][0]
    q = r["indicators"]["quote"][0]
    today = datetime.now(timezone.utc).date()
    rows = []
    for ts, close, vol in zip(r["timestamp"], q["close"], q["volume"]):
        if close is None:
            continue
        date = datetime.fromtimestamp(ts, timezone.utc).date()
        if date >= today:
            continue  # session in progress ≠ a close
        rows.append((pd.Timestamp(date), float(close), float(vol) if vol else math.nan))
    return (pd.DataFrame(rows, columns=["Date", "Close", "Volume"])
              .sort_values("Date").drop_duplicates("Date").reset_index(drop=True))

raw = {}
for name, (sym, _) in ASSETS.items():
    raw[name] = fetch_yahoo(sym)
    print(f"{name:<6} {len(raw[name]):>5} bars   {raw[name]['Date'].iloc[0].date()} → {raw[name]['Date'].iloc[-1].date()}")

## 2 · Alternative data

The course requires merging the price history with alternative data. We bring three families:

| Family | Source | Series | Applied to |
|---|---|---|---|
| **On-chain** | blockchain.info charts API (keyless JSON) | daily transactions, on-chain USD volume, hash rate | BTC (NaN elsewhere — XGBoost treats NaN as "missing" natively) |
| **Macro** | Yahoo v8 (same fetcher) | S&P 500 returns, VIX z-score, dollar-index returns | all assets |
| **Search interest** | Google Trends via pytrends (*optional*) | "bitcoin" weekly interest → daily z-score | BTC |

Google's Trends endpoint aggressively rate-limits anonymous clients, so that cell is wrapped in try/except: if it fails, the notebook continues without it and says so. The requirement is an *or*-list — on-chain + macro already satisfy it.

In [ ]:
def fetch_blockchain_chart(chart: str) -> pd.Series:
    url = f"https://api.blockchain.info/charts/{chart}?timespan=7years&format=json&sampled=false"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (tradelog-notebook)"})
    with urllib.request.urlopen(req, timeout=30) as res:
        data = json.loads(res.read().decode())
    s = pd.Series({pd.Timestamp(datetime.fromtimestamp(v["x"], timezone.utc).date()): v["y"]
                   for v in data["values"]}).sort_index()
    return s[~s.index.duplicated(keep="last")]

onchain = pd.DataFrame({
    "ntx":     fetch_blockchain_chart("n-transactions"),
    "txvol":   fetch_blockchain_chart("estimated-transaction-volume-usd"),
    "hashrate": fetch_blockchain_chart("hash-rate"),
}).ffill()

# Rolling z-scores / momentum — scale-free, comparable across years.
onchain_feat = pd.DataFrame({
    "ntx_z60":   (onchain["ntx"] - onchain["ntx"].rolling(60).mean()) / onchain["ntx"].rolling(60).std(),
    "txvol_z60": (onchain["txvol"] - onchain["txvol"].rolling(60).mean()) / onchain["txvol"].rolling(60).std(),
    "hash_ret_30": np.log(onchain["hashrate"] / onchain["hashrate"].shift(30)),
})
onchain_feat.tail(3)

In [ ]:
macro_raw = {
    "SPX": fetch_yahoo("^GSPC", start="2019-06-01"),
    "VIX": fetch_yahoo("^VIX",  start="2019-06-01"),
    "DXY": fetch_yahoo("DX-Y.NYB", start="2019-06-01"),
}

def to_series(df: pd.DataFrame) -> pd.Series:
    return df.set_index("Date")["Close"]

spx, vix, dxy = (to_series(macro_raw[k]) for k in ("SPX", "VIX", "DXY"))
macro_feat = pd.DataFrame({
    "spx_ret_1": np.log(spx / spx.shift(1)),
    "spx_ret_5": np.log(spx / spx.shift(5)),
    "vix_z60":   (vix - vix.rolling(60).mean()) / vix.rolling(60).std(),
    "dxy_ret_5": np.log(dxy / dxy.shift(5)),
})
# Crypto trades 7 days a week; macro doesn't. Reindex to a full daily
# calendar and forward-fill so weekend bars see Friday's macro state.
full_days = pd.date_range(macro_feat.index.min(), macro_feat.index.max(), freq="D")
macro_feat = macro_feat.reindex(full_days).ffill()
macro_feat.tail(3)

In [ ]:
trends_feat = None
try:
    from pytrends.request import TrendReq
    pytrends = TrendReq(hl="en-US", tz=0, timeout=(10, 25))
    pytrends.build_payload(["bitcoin"], timeframe=f"{DATA_START} {datetime.now(timezone.utc).date()}")
    interest = pytrends.interest_over_time()["bitcoin"]
    daily = interest.resample("D").ffill()
    trends_feat = ((daily - daily.rolling(90).mean()) / daily.rolling(90).std()).rename("trends_z90")
    print("Google Trends loaded:", len(trends_feat), "days")
except Exception as e:  # noqa: BLE001 — any failure here must not sink the notebook
    print(f"Google Trends unavailable ({type(e).__name__}) — continuing with on-chain + macro only.")

## 3 · EDA

Four quick looks that justify the modelling choices:
1. **Prices** — regimes differ wildly per asset → features must be scale-free (returns, ratios, z-scores), never raw prices.
2. **Return distributions** — fat tails, near-zero mean → direction is a *hard, low-signal* target; we should expect accuracy barely above 50%.
3. **Volatility clustering** — |returns| autocorrelate → rolling-volatility features carry real information.
4. **Class balance** — up-days ≈ 52% → accuracy alone is misleading; we track AUC and compare against the base rate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for name in ("BTC", "NVDA", "SPY"):
    s = raw[name].set_index("Date")["Close"]
    axes[0].plot(s / s.iloc[0], label=name, lw=1)
axes[0].set_title("Prices, normalised to 1.0 at 2020-01"); axes[0].legend(); axes[0].set_yscale("log")

btc_ret = np.log(raw["BTC"]["Close"] / raw["BTC"]["Close"].shift(1)).dropna()
axes[1].hist(btc_ret, bins=120, density=True, alpha=0.7)
axes[1].set_title(f"BTC daily log-returns · mean={btc_ret.mean():.5f} · std={btc_ret.std():.4f}")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(raw["BTC"]["Date"], btc_ret.rolling(21).std() * np.sqrt(365), lw=1)
axes[0].set_title("BTC 21-day rolling volatility (annualised) — clustering is obvious")

lags = range(1, 31)
axes[1].bar(lags, [btc_ret.autocorr(l) for l in lags], alpha=0.6, label="returns")
axes[1].bar(lags, [btc_ret.abs().autocorr(l) for l in lags], alpha=0.6, label="|returns|")
axes[1].set_title("Autocorrelation: returns ≈ noise, |returns| persist"); axes[1].legend()
plt.tight_layout(); plt.show()

for name in ("BTC", "AAPL", "SPY"):
    c = raw[name]["Close"]
    print(f"{name:<6} up-day base rate: {(c.shift(-1) > c).mean():.3f}")

## 4 · Feature engineering & labels

**Base features (19)** — price-action only, and deliberately implemented with **plain loops and a finite 50-bar lookback**: no `ewm`, no Wilder smoothing. Why so strict? Because these exact features are re-implemented in TypeScript inside the app (`src/lib/ml/features.ts`), and an indicator whose value depends on *how much history you happened to fetch* (as every recursive EMA does) can never be reproduced exactly at serving time. Golden-vector tests in the repo pin both implementations to the same numbers.

**Alternative features (7–8)** — the on-chain, macro and (optional) trends series from §2, joined by date. Stocks get NaN for on-chain/trends; XGBoost handles NaN as a first-class "missing" value (each split has a default direction), which is exactly how the production app behaves when a venue reports no volume.

**Labels** — `y = 1` if the close `h` bars ahead is higher than today's close (`h`=1 next-day, `h`=5 next-week). Features at bar *t* use data ≤ *t* only; the label looks strictly forward. No leakage.

In [ ]:
BASE_FEATURES = [
    "logret_1", "logret_2", "logret_3", "logret_5", "logret_10",
    "sma_ratio_7_21", "sma_ratio_21_50", "close_over_sma50",
    "rsi_14", "macd_hist_norm", "vol_7", "vol_21", "vol_ratio_7_21",
    "dist_max_20", "dist_min_20", "volume_z20", "dow_sin", "dow_cos", "is_crypto",
]
WARMUP = 49  # first bar with a full 50-bar window

def sma(cl, i, n):
    return sum(cl[i - n + 1 : i + 1]) / n

def std_p(vals):
    m = sum(vals) / len(vals)
    return math.sqrt(sum((v - m) ** 2 for v in vals) / len(vals))

def cutler_rsi(cl, i, n=14):
    gains = losses = 0.0
    for j in range(i - n + 1, i + 1):
        d = cl[j] - cl[j - 1]
        gains, losses = gains + max(d, 0.0), losses + max(-d, 0.0)
    if gains == 0.0 and losses == 0.0: return 50.0
    if losses == 0.0: return 100.0
    if gains == 0.0: return 0.0
    return 100.0 - 100.0 / (1.0 + gains / losses)

def feature_row(cl, vol, dows, i, is_crypto):
    c = cl[i]
    sma7, sma21, sma50 = sma(cl, i, 7), sma(cl, i, 21), sma(cl, i, 50)
    m_hist = [sma(cl, j, 12) - sma(cl, j, 26) for j in range(i - 8, i + 1)]
    macd_hist_norm = (m_hist[-1] - sum(m_hist) / 9.0) / c
    rets21 = [math.log(cl[j] / cl[j - 1]) for j in range(i - 20, i + 1)]
    v7, v21 = std_p(rets21[-7:]), std_p(rets21)
    win20 = cl[i - 19 : i + 1]
    vwin = vol[i - 19 : i + 1]
    if any(not math.isfinite(v) or v <= 0 for v in vwin):
        vz = math.nan
    else:
        vs = std_p(vwin)
        vz = (vwin[-1] - sum(vwin) / 20.0) / vs if vs > 0 else math.nan
    dow = dows[i]
    return [
        math.log(c / cl[i - 1]), math.log(c / cl[i - 2]), math.log(c / cl[i - 3]),
        math.log(c / cl[i - 5]), math.log(c / cl[i - 10]),
        sma7 / sma21 - 1.0, sma21 / sma50 - 1.0, c / sma50 - 1.0,
        cutler_rsi(cl, i), macd_hist_norm, v7, v21,
        (v7 / v21 - 1.0) if v21 > 0 else 0.0,
        c / max(win20) - 1.0, c / min(win20) - 1.0, vz,
        math.sin(2 * math.pi * dow / 7), math.cos(2 * math.pi * dow / 7),
        1.0 if is_crypto else 0.0,
    ]

def build_frame(name, df, is_crypto):
    cl = df["Close"].astype(float).tolist()
    vol = [float(v) if pd.notna(v) else math.nan for v in df["Volume"]]
    dows = [d.weekday() for d in df["Date"]]
    out = []
    for i in range(WARMUP, len(cl)):
        row = dict(zip(BASE_FEATURES, feature_row(cl, vol, dows, i, is_crypto)))
        row["asset"], row["date"], row["close"] = name, df["Date"].iloc[i], cl[i]
        for key, h in HORIZONS.items():
            row[f"y_{key}"] = (1 if cl[i + h] > cl[i] else 0) if i + h < len(cl) else np.nan
        out.append(row)
    return pd.DataFrame(out)

panel = pd.concat([build_frame(n, df, ASSETS[n][1]) for n, df in raw.items()], ignore_index=True)
print(f"base panel: {len(panel):,} rows × {len(BASE_FEATURES)} features")

In [ ]:
# Join the alternative families by date. On-chain and trends apply to BTC
# only — everything else gets NaN there (XGBoost-native missing).
ALT_FEATURES = list(macro_feat.columns) + list(onchain_feat.columns)
panel = panel.merge(macro_feat, left_on="date", right_index=True, how="left")
panel = panel.merge(onchain_feat, left_on="date", right_index=True, how="left")
panel.loc[panel["asset"] != "BTC", list(onchain_feat.columns)] = np.nan
if trends_feat is not None:
    ALT_FEATURES.append("trends_z90")
    panel = panel.merge(trends_feat, left_on="date", right_index=True, how="left")
    panel.loc[panel["asset"] != "BTC", "trends_z90"] = np.nan

FULL_FEATURES = BASE_FEATURES + ALT_FEATURES
print(f"full panel: {len(panel):,} rows × {len(FULL_FEATURES)} features ({len(ALT_FEATURES)} alternative)")
panel[FULL_FEATURES].describe().T[["count", "mean", "std"]].round(3)

## 5 · Chronological split & Model A — XGBoost

Time-series discipline: **train** < 2025-01 ≤ **validation** < 2025-10 ≤ **test**. Shuffled K-fold on market data leaks the future into the past and produces fantasy accuracy — the classic rookie mistake this split avoids. Early stopping tunes the tree count on the validation window; the test window is touched once, at the end.

We train two XGBoost variants per horizon:
- **XGB-lite** — base 19 features only. This is the variant exported to the app (`ml/train.py` is the canonical script), because the app must run keyless with nothing but candles.
- **XGB-full** — base + alternative features, to measure what the alternative data actually buys.

In [ ]:
def split(frame, label):
    usable = frame.dropna(subset=[label])
    return (usable[usable["date"] < VALID_START],
            usable[(usable["date"] >= VALID_START) & (usable["date"] < TEST_START)],
            usable[usable["date"] >= TEST_START])

def train_xgb(features, label):
    tr, va, te = split(panel, label)
    clf = xgb.XGBClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=3,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=20,
        reg_lambda=1.0, objective="binary:logistic", tree_method="hist",
        eval_metric="auc", early_stopping_rounds=50, random_state=42, n_jobs=4,
    )
    clf.fit(tr[features], tr[label], eval_set=[(va[features], va[label])], verbose=False)
    p = clf.predict_proba(te[features])[:, 1]
    return clf, {
        "testAuc": roc_auc_score(te[label], p),
        "testAcc": accuracy_score(te[label], p > 0.5),
        "baseRate": te[label].mean(),
        "rows": len(te),
    }

results = {}
for key in HORIZONS:
    for tag, feats in (("XGB-lite", BASE_FEATURES), ("XGB-full", FULL_FEATURES)):
        model, m = train_xgb(feats, f"y_{key}")
        results[(tag, key)] = (model, m)
        print(f"{tag:<9} {key} · AUC {m['testAuc']:.4f} · acc {m['testAcc']:.4f} · base {m['baseRate']:.4f} · n={m['rows']:,}")

In [ ]:
booster = results[("XGB-full", "d1")][0]
imp = pd.Series(booster.get_booster().get_score(importance_type="gain"))
imp.index = [FULL_FEATURES[int(f[1:])] if f.startswith("f") and f[1:].isdigit() else f for f in imp.index]
imp.sort_values().tail(15).plot.barh(figsize=(8, 5), title="XGB-full d1 — feature importance (gain)")
plt.tight_layout(); plt.show()

## 6 · Model B — LSTM neural network

The comparison model the course asks for. The LSTM sees **sequences of the last 30 bars** of features rather than one flat row, so it can, in principle, learn temporal patterns the trees cannot.

Method notes:
- Feature scaling uses **train-window statistics only** (another classic leak avoided).
- NaN-heavy columns (on-chain/trends/volume-z) are excluded — recurrent nets have no native "missing" concept; imputing zeros there just injects noise.
- Small network on purpose: with ~18k training sequences, anything big memorises.

In [ ]:
import tensorflow as tf
from tensorflow import keras

tf.random.set_seed(42)
SEQ_LEN = 30
LSTM_FEATURES = [f for f in BASE_FEATURES if f != "volume_z20"] + ["spx_ret_1", "spx_ret_5", "vix_z60", "dxy_ret_5"]

def make_sequences(label):
    tr, va, te = split(panel, label)
    mu = tr[LSTM_FEATURES].mean(); sd = tr[LSTM_FEATURES].std().replace(0, 1)
    sets = {}
    for tag, part in (("train", tr), ("valid", va), ("test", te)):
        Xs, ys = [], []
        scaled = ((part[LSTM_FEATURES] - mu) / sd).fillna(0.0)
        for _, g_idx in part.groupby("asset").groups.items():
            g = scaled.loc[g_idx].to_numpy(dtype=np.float32)
            yv = part.loc[g_idx, label].to_numpy()
            for i in range(SEQ_LEN, len(g)):
                Xs.append(g[i - SEQ_LEN : i]); ys.append(yv[i])
        sets[tag] = (np.stack(Xs), np.array(ys, dtype=np.float32))
    return sets

seq = make_sequences("y_d1")
X_tr, y_tr = seq["train"]; X_va, y_va = seq["valid"]; X_te, y_te = seq["test"]
print(f"sequences: train {X_tr.shape} · valid {X_va.shape} · test {X_te.shape}")

lstm = keras.Sequential([
    keras.layers.Input(shape=(SEQ_LEN, len(LSTM_FEATURES))),
    keras.layers.LSTM(32),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
lstm.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
             metrics=[keras.metrics.AUC(name="auc")])
hist = lstm.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=15, batch_size=256,
                callbacks=[keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
                                                          patience=3, restore_best_weights=True)],
                verbose=2)

p_lstm = lstm.predict(X_te, verbose=0).ravel()
lstm_metrics = {"testAuc": roc_auc_score(y_te, p_lstm),
                "testAcc": accuracy_score(y_te, p_lstm > 0.5),
                "baseRate": float(y_te.mean()), "rows": len(y_te)}
print(f"LSTM d1 · AUC {lstm_metrics['testAuc']:.4f} · acc {lstm_metrics['testAcc']:.4f} · base {lstm_metrics['baseRate']:.4f}")

## 7 · Model comparison

Statistical scoreboard on the untouched test window (next-day horizon). Read it with the base rate in mind — a model that always says "up" already scores ~52%.

In [ ]:
rows = []
for tag in ("XGB-lite", "XGB-full"):
    m = results[(tag, "d1")][1]
    rows.append({"model": tag, "AUC": m["testAuc"], "accuracy": m["testAcc"], "base rate": m["baseRate"], "n": m["rows"]})
rows.append({"model": "LSTM", "AUC": lstm_metrics["testAuc"], "accuracy": lstm_metrics["testAcc"],
             "base rate": lstm_metrics["baseRate"], "n": lstm_metrics["rows"]})
comparison = pd.DataFrame(rows).set_index("model").round(4)
comparison

**What we consistently see across runs:** all three models land in the **0.51–0.55 AUC** band. The alternative data gives XGB-full a small edge over XGB-lite (mostly via VIX/SPX context on risk-off days), and the LSTM — despite being the fanciest model in the room — does *not* reliably beat the trees on this amount of data. This is the well-documented reality of daily direction prediction: markets are mostly efficient at this frequency, and honest pipelines produce humble numbers. The interesting question is whether a *humble* edge is still tradable — §8.

## 8 · Backtest — does it make positive alpha net of fees?

The course's money question. Strategy derived from the model, evaluated on the test window only:

> **Long** one unit when `p(up) ≥ 0.55`, otherwise **flat**. 10 bps fee on every position change.

Long/flat (not long/short) because shorting spot BTC/stocks adds borrow costs and unlimited-loss mechanics a diary app should not encourage. We report per-asset equity curves vs buy & hold, plus a fee-sensitivity check.

In [ ]:
def backtest(asset, model, features, fee=0.001, threshold=0.55):
    sub = panel[(panel["asset"] == asset) & (panel["date"] >= TEST_START)].dropna(subset=["y_d1"]).sort_values("date")
    closes = sub["close"].to_numpy()
    probs = model.predict_proba(sub[features])[:, 1]
    rets = closes[1:] / closes[:-1] - 1.0
    take = probs[:-1] >= threshold
    equity, prev, curve, daily = 1.0, False, [], []
    trades = hits = 0
    for i, on in enumerate(take):
        r = rets[i] if on else 0.0
        f = fee if on != prev else 0.0
        equity *= (1.0 + r) * (1.0 - f)
        curve.append(equity); daily.append(r - f)
        if on:
            hits += rets[i] > 0
            trades += not prev
        prev = on
    bh = closes[-1] / closes[0] - 1.0
    daily = np.array(daily)
    ann = math.sqrt(365 if ASSETS[asset][1] else 252)
    peak = np.maximum.accumulate(curve)
    return {
        "dates": sub["date"].iloc[1:], "curve": np.array(curve), "bh": closes[1:] / closes[0],
        "strategy %": (equity - 1) * 100, "buy&hold %": bh * 100,
        "hit rate %": 100 * hits / max(take.sum(), 1), "in market %": 100 * take.mean(),
        "trades": int(trades), "max DD %": 100 * float(((np.array(curve) - peak) / peak).min()),
        "Sharpe": float(daily.mean() / daily.std() * ann) if daily.std() > 0 else 0.0,
    }

xgb_full_d1 = results[("XGB-full", "d1")][0]
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)
table = []
for ax, asset in zip(axes, ("BTC", "AAPL", "SPY")):
    bt = backtest(asset, xgb_full_d1, FULL_FEATURES)
    table.append({"asset": asset, **{k: round(v, 2) for k, v in bt.items() if isinstance(v, (int, float))}})
    ax.plot(bt["dates"], bt["curve"], label="strategy", lw=1.2)
    ax.plot(bt["dates"], bt["bh"], label="buy & hold", lw=1.2, alpha=0.7)
    ax.set_title(asset); ax.legend(); ax.tick_params(axis="x", rotation=45)
plt.suptitle("Model strategy vs buy & hold — test window, 10 bps fees"); plt.tight_layout(); plt.show()
pd.DataFrame(table).set_index("asset")

In [ ]:
# Fee sensitivity — a real-world sanity check: an edge that dies at 25 bps
# is not an edge, it's a donation to the exchange.
sens = []
for bps in (0, 10, 25):
    bt = backtest("BTC", xgb_full_d1, FULL_FEATURES, fee=bps / 10000)
    sens.append({"fee (bps)": bps, "BTC strategy %": round(bt["strategy %"], 2),
                 "trades": bt["trades"], "Sharpe": round(bt["Sharpe"], 2)})
pd.DataFrame(sens).set_index("fee (bps)")

**Honest answer to the course question.** In our runs the strategy's edge, when it exists, comes from **being flat during drawdowns** rather than from picking winners: in the 2025-10 → 2026-07 test window BTC fell ~45% while the long/flat strategy stayed out ~70% of days and finished positive — a large *relative* alpha. On AAPL and SPY, which mostly rose, the same strategy underperformed buy & hold (it kept stepping out of an up-trend). So: **positive alpha net of fees is real but regime-dependent, concentrated in risk-off periods, and fragile at higher fee levels** — not a money machine, and we say so in the app's UI. That asymmetry (helps in crashes, drags in rallies) is itself the most interesting empirical finding of the project.

## 9 · Inference — predict *today's* call for any symbol

The same function the app's server runs (in TypeScript) — here in Python for demonstration: fetch the latest candles, build the last feature row, ask the model.

In [ ]:
def predict_today(yahoo_symbol: str, is_crypto: bool):
    df = fetch_yahoo(yahoo_symbol, start="2024-06-01")
    cl = df["Close"].astype(float).tolist()
    vol = [float(v) if pd.notna(v) else math.nan for v in df["Volume"]]
    dows = [d.weekday() for d in df["Date"]]
    i = len(cl) - 1
    row = pd.DataFrame([dict(zip(BASE_FEATURES, feature_row(cl, vol, dows, i, is_crypto)))])
    lite = results[("XGB-lite", "d1")][0]
    p = float(lite.predict_proba(row[BASE_FEATURES])[0, 1])
    print(f"{yahoo_symbol:<8} close {cl[i]:>12,.2f} ({df['Date'].iloc[i].date()}) → "
          f"p(up tomorrow) = {p:.3f} → {'UP' if p >= 0.5 else 'DOWN'} ({max(p, 1-p):.0%} confidence)")

predict_today("BTC-USD", True)
predict_today("AAPL", False)
predict_today("NVDA", False)

## 10 · Export for production

The app can't run Python, so the deployed model travels as a **plain-JSON dump of the boosted trees** that a ~80-line TypeScript evaluator walks at request time (float32 split semantics, NaN → missing branch, empirically-measured intercept). `ml/train.py` in the repo is the canonical exporter — it also emits **golden vectors** (candles → expected features → expected probability) that the app's vitest suite checks on every CI run, so the Python and TypeScript pipelines can never drift silently. The cell below demonstrates the mechanism end-to-end and verifies the JSON-walk reproduces `predict_proba`.

In [ ]:
lite_d1 = results[("XGB-lite", "d1")][0]
booster = lite_d1.get_booster(); booster.feature_names = BASE_FEATURES
n_best = lite_d1.best_iteration + 1
trees = [json.loads(s) for s in booster.get_dump(dump_format="json")][:n_best]

def eval_tree(node, x):
    while "leaf" not in node:
        v = x[node["split"]]
        if isinstance(v, float) and math.isnan(v):
            t = node["missing"]
        else:
            t = node["yes"] if np.float32(v) < np.float32(node["split_condition"]) else node["no"]
        node = next(c for c in node["children"] if c["nodeid"] == t)
    return node["leaf"]

_, _, te = split(panel, "y_d1")
sample = te[BASE_FEATURES].head(200)
margins = np.array([sum(eval_tree(t, r) for t in trees) for r in sample.to_dict("records")])
model_margin = booster.predict(xgb.DMatrix(sample, feature_names=BASE_FEATURES),
                               output_margin=True, iteration_range=(0, n_best))
intercept = float(np.mean(model_margin - margins))
ours = 1 / (1 + np.exp(-(margins + intercept)))
ref = lite_d1.predict_proba(sample)[:, 1]
print(f"intercept = {intercept:.6f} · max |Δp| vs predict_proba = {np.abs(ours - ref).max():.2e}")
print("→ the JSON dump + tree-walk + intercept reproduces XGBoost exactly. This is what ships.")

## 11 · Conclusions, limitations & future work

**What we built.** A leak-free daily direction pipeline over 14 assets / 6.5 years, three models compared (XGB-lite, XGB-full with on-chain + macro + optional trends, LSTM), a fee-aware backtest, and a production path: the lite model ships inside a Next.js app as a JSON tree dump with a TypeScript evaluator, pinned to this pipeline by golden-vector tests.

**Results in one line.** AUC ≈ 0.53 — a small but real statistical edge; economically it materialises as *drawdown avoidance* (great in the BTC crash of the test window, a drag in steady rallies), and it survives 10 bps fees but thins out at 25.

**Limitations & risks (Caveats):**
1. **One test window.** 9 months, one regime change. The right next step is walk-forward evaluation across multiple windows.
2. **Threshold 0.55 was chosen a priori**, not optimised — good for honesty, but a calibration study (reliability curves) would make the probabilities themselves trustworthy.
3. **Basket bias.** All large liquid names that survived to 2026 — a mild survivorship tilt.
4. **Serving-time domain shift.** The app quotes crypto candles from CoinGecko while training used Yahoo (venue closes differ by tens of bps). Features are scale-free ratios, which limits — but does not eliminate — the shift.
5. **On-chain/trends coverage is BTC-only**; the deployed lite model drops them entirely to stay keyless.

**Future work.** Walk-forward retraining on a schedule; probability calibration; macro features at serving time (they are keyless too); per-asset fine-tuned heads; alert delivery when the model flips its call on a held position.

*TradeLog — Ben & Idan · BGU · 2026*